In [5]:
##############
## INITIATE ##
##############

## Imports
import os
import pandas as pd
import pickle
import numpy as np

## Import modules
import p_pawmil as pawmil

#
###

In [8]:
## Lists
MODEL_NAME_LIST = [    
    # 'model_resnet_tcgaoct_raw_FTL_1',
    # 'model_resnet_tcgaoct_raw_FTL_3',
    # 'model_resnet_tcgaoct_raw_FTL_5',
    # 'model_resnet_tcgaoct_raw_FTL_25',
    'model_resnet_icgcc_raw_FTL_25',
]

# ## Lists
# MODEL_NAME_LIST = [
#     'model_resnet_icgcc_baseline',
#     'model_resnet_icgcc_raw_FTL_0',
#     'model_resnet50_icgcc_baseline', 
#     'model_resnet50_icgcc_raw_FTL_0', 
#     'model_vit_icgcc_baseline', 
#     'model_vit_icgcc_raw_FTL_0', 
# ]


for MODEL_NAME in MODEL_NAME_LIST:

    for AGGREGATOR in ['pawmil', 'clam']:

        ############################
        ## INITIATE HYPERTRAINING ##
        ############################
        
        ## User inputs
        # MODEL_NAME = 'model_vit_icgcc_raw_FTL_0'
        COHORT = 'icgc-c'
        # AGGREGATOR = 'clam'
        
        ## CSV variables (ICGCC)
        if COHORT == 'icgc-c':
            pti = '/Volumes/Elements/BDI/projects/GSG/metadata/icgcc-relapse/metadata_ICGCCRelapse.csv' # ICGCC 
            path_column = 'path_to_image_1' # ICGCC
            case_id_column = 'slide_id' # ICGCC
            center_column = 'originating_lab' # ICGCC
        
        ## CSV variables (TCGA)
        if COHORT == 'tcga-oct':
            pti = '/Volumes/Elements/BDI/projects/GSG/metadata/tcga-cnv/master-sheet-2025-04-18.csv' # TCGA
            path_column = 'wsi_file_path_local' # TCGA
            case_id_column = 'sample_id_local' # TCGA
            center_column = 'Project ID' # TCGA
        
        ## Load the CSV file
        df = pd.read_csv(pti)
        print(df.shape)
        
        ## Define the paths to the input files
        pt_embeddings_file = f"/Volumes/Elements/BDI/projects/GSG/outputs/o3_mil/{COHORT}/{MODEL_NAME}/encoder/obj_embeddings.pkl"
        PTO = f"/Volumes/Elements/BDI/projects/GSG/outputs/o3_mil/{COHORT}/{MODEL_NAME}/"
        
        ## Define the columns to extract
        all_labels_columns = [
            'relapse',
            # 'EMT.GAIN',
            # 'GAIN.ZEB1',
            # 'GAIN.ZEB2',
            # 'GAIN.TWIST1',
            # 'GAIN.TWIST2',
            # 'GAIN.SNAI1',
            # 'GAIN.SNAI2',
            # 'MYC.GAIN',
            # 'PTEN.GAIN',
            # 'TP53.LOSS',
            # 'RB1.LOSS',	
            # 'CHD1.LOSS',	
            # 'FOXA1.GAIN',	
            # 'NKX3-1.LOSS',	
            # 'CDKN1B.LOSS',	
            # 'TMPRSS2.GAIN',	
            # 'ERG.GAIN',
        ]
        
        #
        ###
    
        #############################
        ## SINGLE LABEL HYPERTRAIN ##
        #############################
        
        ## Hypertrain labels sequentially
        for selected_label_column in all_labels_columns:
        
            ## Updater
            print("")
            print(f"Hypertraining singletask-{selected_label_column}")
        
            ## Create output directory
            pto = PTO + f"aggregator_{AGGREGATOR}_1000/singletask-{selected_label_column}/"
            if os.path.exists(pto) == False:
                os.makedirs(pto, exist_ok=True)
            
            ## Extract the relevant columns
            labels = df[[selected_label_column]]
            paths = df[[path_column]]
            case_ids = df[[case_id_column]]
            centers = df[[center_column]]
            
            ## Threshold the labels to binarize them
            labels = (labels > 0.0).astype(int)
            
            ## Repeat each row n times to match the augmented dataset
            n = 1
            labels_repeated = pd.concat([labels] * n, ignore_index=True)
            paths_repeated = pd.concat([paths] * n, ignore_index=True)
            case_ids_repeated = pd.concat([case_ids] * n, ignore_index=True)
            centers_repeated = pd.concat([centers] * n, ignore_index=True)
            
            ## Save the extracted and repeated columns to separate text files with tab separation
            labels_repeated.to_csv(f"{pto}/tab_labels.txt", index=False, header=False, sep='\t')
            paths_repeated.to_csv(f"{pto}/tab_paths.txt", index=False, header=False, sep='\t')
            case_ids_repeated.to_csv(f"{pto}/tab_groups.txt", index=False, header=False, sep='\t')
            centers_repeated.to_csv(f"{pto}/tab_groups_2.txt", index=False, header=False, sep='\t')
              
            ## Hypertrain aggregator
            for i in [1, 2, 3, 4, 5]:
                if AGGREGATOR == 'clam':
                    pawmil.hypertrain_aggregator_clam(
                        pt_embeddings_file = pt_embeddings_file, 
                        pt_labels_file = f"{pto}/tab_labels.txt",
                        pt_groups_file = f"{pto}/tab_groups.txt",
                        pt_output_folder = pto,
                        selected_testing_fold = i,
                        device = "mps", 
                        hp_list = [[1], [4, 8], [0.0001, 0.001]]
                        )
                else:
                    pawmil.hypertrain_aggregator(
                        pt_embeddings_file = pt_embeddings_file, 
                        pt_labels_file = f"{pto}/tab_labels.txt",
                        pt_groups_file = f"{pto}/tab_groups.txt",
                        pt_output_folder = pto,
                        selected_testing_fold = i,
                        device = "mps", 
                        hp_list = [[1], [4, 8], [0.0001, 0.001]]
                        )
        
            ## End
            print("")
            print("Hypertraining completed")
        
        #
        ###

(393, 44)

Hypertraining singletask-relapse
Fold: 1, Repeat: 1, Hyperparameters:  1,  4, 0.0001, Train auc: 0.86, Validation auc: 0.84, Test auc: 0.79
Fold: 1, Repeat: 2, Hyperparameters:  1,  4, 0.0001, Train auc: 0.77, Validation auc: 0.78, Test auc: 0.69
Fold: 1, Repeat: 3, Hyperparameters:  1,  4, 0.0001, Train auc: 0.83, Validation auc: 0.82, Test auc: 0.72
Fold: 2, Repeat: 1, Hyperparameters:  1,  4, 0.0001, Train auc: 0.88, Validation auc: 0.79, Test auc: 0.77
Fold: 2, Repeat: 2, Hyperparameters:  1,  4, 0.0001, Train auc: 0.84, Validation auc: 0.74, Test auc: 0.72
Fold: 2, Repeat: 3, Hyperparameters:  1,  4, 0.0001, Train auc: 0.82, Validation auc: 0.74, Test auc: 0.72
Fold: 3, Repeat: 1, Hyperparameters:  1,  4, 0.0001, Train auc: 0.85, Validation auc: 0.82, Test auc: 0.71
Fold: 3, Repeat: 2, Hyperparameters:  1,  4, 0.0001, Train auc: 0.85, Validation auc: 0.84, Test auc: 0.74
Fold: 3, Repeat: 3, Hyperparameters:  1,  4, 0.0001, Train auc: 0.85, Validation auc: 0.83, Test auc